|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Guided decoding<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: make invalid output unreachable<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
from tests.helpers import json_prefix_state
rng = np.random.default_rng(0)
# A vocabulary of multi-character pieces, as in a real tokenizer.
VOCAB = ['{', '}', '[', ']', '"', ':', ',', ' ', '1', '2', '42',
         'true', 'false', 'null', 'name', 'age', '": ', '", "', ': {', '}, ']
print(f'{len(VOCAB)} tokens, and most of them are more than one character')

Constrain the output, so that invalid JSON is unreachable and not only
improbable.

You get the grammar. `json_prefix_state` classifies a string as `valid`,
`prefix` or `invalid`. Your work is the part that touches the sampler, and the
part that makes the mask affordable.

# Exercise 1: which tokens are legal here?

Note that the vocabulary contains multi-character pieces like `'": '`. That
is what makes real guided decoding fiddly: the FSM advances by a whole token,
not a character.

In [ ]:
def legal_mask(prefix):
  """Which tokens can follow `prefix`, and keep it recoverable?
  A token is legal when prefix + its WHOLE string is in state 'prefix' or
  'valid'. Not one character: the whole token."""
  return 

for prefix in ['', '{', '{"', '{"name', '{"name"', '{"name": ']:
  mask = legal_mask(prefix)
  allowed = [token for token, is_legal in zip(VOCAB, mask) if is_legal]
  print(f'{prefix!r:<12} {mask.sum():>2}/{len(VOCAB)}: {allowed[:8]}')

# Exercise 2: sample with the mask on

Two hundred runs with random logits. Not one of them should produce invalid
JSON.

In [ ]:
def masked_sample(prefix, logits):
  """Sample a token, but only from the legal tokens. -> None at a dead end."""
  mask = legal_mask(prefix)
  if not mask.any():
    return None                      # a dead end: see Exercise 3
  # Set the illegal logits to -inf, then softmax, then draw.
  masked = 
  probs = 
  return VOCAB[int(rng.choice(len(VOCAB), p=probs))]

def generate(max_tokens, stop_at_valid):
  """Sample up to max_tokens masked tokens from random logits. -> the text."""
  text = ''
  for _ in range(max_tokens):
    token = masked_sample(text, rng.normal(size=len(VOCAB)))
    if token is None:
      break
    text += token
    if stop_at_valid and json_prefix_state(text) == 'valid' and len(text) > 6:
      break
  return text

invalid_count = 0
for trial in range(200):
  text = generate(max_tokens=14, stop_at_valid=True)
  if json_prefix_state(text) == 'invalid':
    invalid_count += 1
print(f'{invalid_count}/200 samples produced invalid JSON')
print(f'example: {text!r}  ({json_prefix_state(text)})')

# Exercise 3: where the one-step mask is not enough

Count the runs that reach a prefix with no legal continuation at all.

In [ ]:
# The mask makes sure that you never become INVALID. Does it make sure that
# you finish? Run to a length limit, as a server does. Then count the final
# state of each run.
state_counts = {'valid': 0, 'prefix': 0, 'invalid': 0}
examples = []
for trial in range(500):
  

for state, count in state_counts.items():
  print(f'{state:>8}: {count:>4}/500')
print('\nincomplete examples, stopped by the length limit:')
for example in examples:
  print(f'  {example!r}')

### Before you open the solution

1. Exercise 2 gives zero invalid outputs. Exercise 3 must show that most runs
   end in `prefix`, and not in `valid`. State exactly what the mask guarantees
   and what it does not guarantee.
2. What must the masker know to also guarantee completion? Why can a
   string-prefix validator not give that information at a low cost?
3. Your `legal_mask()` runs the validator one time for each vocabulary entry. Take
   151,936 tokens at 2 microseconds each. What does one masked step cost, in
   units of one decode step on your own card? Measure a step and divide.
4. What must you cache? Which value is the key of the cache?